In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score

In [30]:
windows = ["06-08", "10-12", "14-16"]
tissues = ["Neuroblasts", "Neurons", "Glia"]
n_splits = 10

for w in windows:
    
    X = pd.read_csv(f"data/training/hrs{w}/data_diff_hrs{w}.csv", index_col=0)
    
    # Prepare per-window figure
    plt.figure(figsize=(8, 8))
    tissue_plotting_info = {t: {} for t in tissues}

    for t in tissues:
        y_path = f"data/training/hrs{w}/y_{t}.csv"
        if not os.path.exists(y_path):
            continue
        y = pd.read_csv(y_path, index_col=0).iloc[:, 0]

        # Align X and y
        X = X.loc[y.index]

        # Cross-validation
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []
        fold_index = {}

        for (i, (train_idx, test_idx)) in enumerate(kf.split(X)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            fold_index[i] = (train_idx, test_idx)

            classifier = RandomForestClassifier(
                n_estimators=500,
                random_state=0,
                n_jobs=-1
            )
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # ROC curve and AUC
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        fprs, tprs, roc_aucs = [], [], []
        for (true, prob) in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            fprs.append(fpr)
            tprs.append(tpr)
            roc_aucs.append(auc(fpr, tpr))

        # Use linear interpolation to plot mean ROC +/- std
        mean_fpr = np.linspace(0, 1, 100)
        interp_tprs = []

        _, ax = plt.subplots(figsize=(6, 6))

        for idx in range(n_splits):
            interp_tpr = np.interp(mean_fpr, fprs[idx], tprs[idx])
            interp_tpr[0] = 0.0
            interp_tprs.append(interp_tpr)

        mean_tpr = np.mean(interp_tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std(roc_aucs)

        ax.plot(
            mean_fpr,
            mean_tpr,
            label=r"Mean ROC (AUC = %0.3f $\pm$ %0.3f)" % (mean_auc, std_auc),
            lw=1,
            alpha=0.8
        )
        plt.grid(axis='both')
        std_tpr = np.std(interp_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            alpha=0.2,
            label=r"$\pm$ 1 std. dev.",
        )

        ax.set(
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            title=f"ROC - {t}, hrs{w}\nAcc = {mean_acc:.3f} " + r'$\pm$' + f" {std_acc:.3f}",
        )

        plt.plot([0, 1], [0, 1], "k--", lw=1)
        ax.legend(loc="lower right")        
        out_path = f"results/figures/roc_RF_{t}_hrs{w}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close()

        # find and train best (according to ROC AUC) classifier from cross-validation
        best_fold = np.argmax(roc_aucs)
        (train_idx, test_idx) = fold_index[best_fold]
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        cv_best_classifier = RandomForestClassifier(
            n_estimators=500,
            random_state=0,
            n_jobs=-1
        )
        cv_best_classifier.fit(X_train, y_train)

        # train a separate model, on all of the data
        all_data_classifier = RandomForestClassifier(
            n_estimators=500,
            random_state=0,
            n_jobs=-1
        )
        all_data_classifier.fit(X, y)

        # SAVE MODELS
        all_data_path = f"results/models/all_data/RF_{t}_{w}.pkl"
        cv_path = f"results/models/cv/RF_{t}_{w}.pkl"

        os.makedirs("results/models/all_data", exist_ok=True)
        os.makedirs("results/models/cv", exist_ok=True)

        pickle.dump(all_data_classifier, open(all_data_path, "wb"))
        pickle.dump(cv_best_classifier, open(cv_path, "wb"))


        # Record info for per-window plot
        for key in ['mean_fpr', 'mean_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            exec(f"tissue_plotting_info['{t}']['{key}'] = {key}")

        print(f"Trained {t} hrs{w}: ROC curve saved at {out_path}")
    
    # Generate per-window common plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=f"Mean ROC curves, hrs{w}",
    )
    for t in tissues:
        
        # decode the dictionary back lol
        for key in ['mean_fpr', 'mean_tpr', 'tprs_upper', 'tprs_lower', 'mean_auc', 'mean_acc', 'std_auc', 'std_acc']:
            exec(f"{key} = tissue_plotting_info['{t}']['{key}']")

        ax.plot(    
            mean_fpr,
            mean_tpr,
            label=f"Mean ROC for {t} (AUC = {mean_auc:.3f} "+ r'$\pm$'+ f" {std_auc:.3f})",
            lw=1,
            alpha=0.8
        )

        plt.grid(axis='both')

        std_tpr = np.std(interp_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            alpha=0.2,
            #label=f"{t} mean "+ r"$\pm$ 1 std. dev.",
        )

    plt.plot([0, 1], [0, 1], "k--", lw=1)
    ax.legend(loc="lower right")        
    out_path = f"results/figures/roc_RF_window_hrs{w}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Saved ROC plot for window {w}: {out_path}")


Trained Neuroblasts hrs06-08: ROC curve saved at results/figures/roc_RF_Neuroblasts_hrs06-08.png
Trained Neurons hrs06-08: ROC curve saved at results/figures/roc_RF_Neurons_hrs06-08.png
Trained Glia hrs06-08: ROC curve saved at results/figures/roc_RF_Glia_hrs06-08.png
Saved ROC plot for window 06-08: results/figures/roc_RF_window_hrs06-08.png
Trained Neuroblasts hrs10-12: ROC curve saved at results/figures/roc_RF_Neuroblasts_hrs10-12.png
Trained Neurons hrs10-12: ROC curve saved at results/figures/roc_RF_Neurons_hrs10-12.png
Trained Glia hrs10-12: ROC curve saved at results/figures/roc_RF_Glia_hrs10-12.png
Saved ROC plot for window 10-12: results/figures/roc_RF_window_hrs10-12.png
Trained Neuroblasts hrs14-16: ROC curve saved at results/figures/roc_RF_Neuroblasts_hrs14-16.png
Trained Neurons hrs14-16: ROC curve saved at results/figures/roc_RF_Neurons_hrs14-16.png
Trained Glia hrs14-16: ROC curve saved at results/figures/roc_RF_Glia_hrs14-16.png
Saved ROC plot for window 14-16: results/

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>

<Figure size 800x800 with 0 Axes>